# 04 · A credit risk model from scratch (pure NumPy)

**Goal:** predict whether a new customer will be **bad** (not pay back), using **logistic regression written by hand in NumPy**, with no ML library.

**The plan**
1. Turn the table into numbers (one-hot encoding)
2. Split into training and test data
3. Scale the features
4. Write the model: sigmoid → loss → gradient descent
5. Train it
6. Test it: accuracy, confusion matrix, precision and recall
7. Look inside: which features push risk up or down?

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/german_credit_clean.csv")
df.shape

(1000, 25)

## 1. Turning the table into numbers
A model can only do math with **numbers**, but columns like `housing` contain text (`"own"`, `"rent"`, ...).
**One-hot encoding** turns one text column into several 0/1 columns, one per category:

| housing | → | housing_own | housing_rent | housing_for free |
|---|---|---|---|---|
| own | | 1 | 0 | 0 |
| rent | | 0 | 1 | 0 |

pandas does this with `pd.get_dummies`. We leave out sex/marital status and nationality on purpose (fairness, see notebook 03).

In [2]:
numeric_features = ["duration", "credit_amount", "installment_rate", "residence_since",
                    "age", "existing_credits", "num_dependents"]
categorical_features = ["checking_status", "credit_history", "purpose", "savings", "employment",
                        "other_debtors", "property", "other_installment_plans", "housing", "job"]

# One-hot encode the text columns; dtype=float gives 0.0/1.0 instead of True/False
X_table = pd.get_dummies(df[numeric_features + categorical_features],
                         columns=categorical_features, dtype=float)

# From here on we only use NumPy arrays
X = X_table.to_numpy()                 # features: one row per customer
y = df["is_bad"].to_numpy(dtype=float) # target: 1 = bad, 0 = good
feature_names = X_table.columns

print("X shape:", X.shape)   # (customers, features)
print("y shape:", y.shape)

X shape: (1000, 53)
y shape: (1000,)


**What I see:** after one-hot encoding we have **53 number columns** for 1,000 customers.

## 2. Splitting into training and test data
We **train** the model on 80% of the customers and **test** it on the 20% it has never seen.
This is like studying with practice exams and then taking a real exam with new questions. Testing on the training data would be cheating.

`rng.permutation(1000)` shuffles the numbers 0–999 randomly. `seed=42` makes the shuffle the same every time, so results are reproducible.

In [3]:
rng = np.random.default_rng(seed=42)
shuffled = rng.permutation(len(X))      # shuffled row numbers
cut = int(0.8 * len(X))                 # 800

train_rows, test_rows = shuffled[:cut], shuffled[cut:]
X_train, X_test = X[train_rows], X[test_rows]   # fancy indexing: pick rows by a list of positions
y_train, y_test = y[train_rows], y[test_rows]

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(f"Bad share in test data: {y_test.mean():.1%}")

Train: (800, 53) | Test: (200, 53)
Bad share in test data: 34.5%


## 3. Scaling the features
`credit_amount` is in the thousands, but `installment_rate` is between 1 and 4. Without scaling, the big numbers would dominate the training.
**Standardizing** turns every column into "how many standard deviations away from the average": `(x - mean) / std`.

⚠️ We calculate mean and std **from the training data only**, and then apply them to the test data. The test data must stay "unseen", otherwise information leaks from the exam into the studying (this is called **data leakage**).

In [4]:
mean = X_train.mean(axis=0)   # axis=0: one mean per column (feature)
std = X_train.std(axis=0)
std[std == 0] = 1             # avoid dividing by zero for a column that is constant in training

# Broadcasting: a (800, 53) array minus a (53,) array -> each column gets its own mean subtracted
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print("Means after scaling (should be ~0):", X_train.mean(axis=0)[:3].round(3))

Means after scaling (should be ~0): [ 0. -0. -0.]


## 4. The model
**Logistic regression** in three ideas:
1. **Score:** every feature gets a **weight** `w`, and the score is `X @ w + b` (a weighted sum, written as a matrix multiplication). Positive weight = pushes towards "bad".
2. **Sigmoid:** squeezes any score into a **probability between 0 and 1**: `sigmoid(z) = 1 / (1 + e^-z)`
3. **Loss:** measures how wrong the probabilities are (binary cross-entropy). Training means **making the loss smaller**.

**Gradient descent** = repeat many times: look at the errors (`p - y`), then move each weight a small step (`learning_rate`) in the direction that lowers the loss.

In [5]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def loss(y_true, p):
    eps = 1e-12   # tiny number so we never take log(0)
    return -np.mean(y_true * np.log(p + eps) + (1 - y_true) * np.log(1 - p + eps))

# Quick check: sigmoid turns any number into a probability
print(sigmoid(np.array([-5, 0, 5])).round(3))

[0.007 0.5   0.993]


## 5. Training

In [6]:
n_features = X_train.shape[1]
w = np.zeros(n_features)   # start with all weights at 0
b = 0.0                    # bias (the starting score)
learning_rate = 0.1

for step in range(2001):
    p = sigmoid(X_train @ w + b)        # 1. predicted probability for all 800 customers at once
    error = p - y_train                 # 2. how wrong each prediction is

    # 3. gradients: how much each weight contributed to the error
    grad_w = X_train.T @ error / len(y_train)
    grad_b = error.mean()

    # 4. take a small step downhill
    w -= learning_rate * grad_w
    b -= learning_rate * grad_b

    if step % 500 == 0:
        print(f"step {step:4d} | loss {loss(y_train, p):.4f}")

step    0 | loss 0.6931
step  500 | loss 0.4380
step 1000 | loss 0.4378
step 1500 | loss 0.4378
step 2000 | loss 0.4378


**What I see:** the loss drops from **0.693** (random guessing) to about **0.438** and then stays flat, so the model has finished learning.

## 6. Testing on unseen customers
A probability ≥ 0.5 means we predict **bad**. Then we compare with the truth.

| | predicted bad | predicted good |
|---|---|---|
| **really bad** | ✅ true positive (TP) | ❌ false negative (FN): **the costly mistake** |
| **really good** | ❌ false positive (FP) | ✅ true negative (TN) |

- **Accuracy** = share of all predictions that are correct
- **Precision** = of the customers we flagged as bad, how many really were bad
- **Recall** = of all really bad customers, how many we caught

In [7]:
def evaluate(threshold):
    p_test = sigmoid(X_test @ w + b)
    pred = (p_test >= threshold).astype(float)

    tp = np.sum((pred == 1) & (y_test == 1))
    fp = np.sum((pred == 1) & (y_test == 0))
    fn = np.sum((pred == 0) & (y_test == 1))
    tn = np.sum((pred == 0) & (y_test == 0))

    print(f"Threshold {threshold}")
    print(f"  Accuracy : {(tp + tn) / len(y_test):.1%}")
    print(f"  Precision: {tp / (tp + fp):.1%}")
    print(f"  Recall   : {tp / (tp + fn):.1%}   (caught {tp} of {tp + fn} bad customers)")
    print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")

baseline = 1 - y_test.mean()   # accuracy of always guessing "good"
print(f"Baseline (always 'good'): {baseline:.1%}\n")
evaluate(0.5)

train_acc = ((sigmoid(X_train @ w + b) >= 0.5) == y_train).mean()
print(f"\nTraining accuracy: {train_acc:.1%}")

Baseline (always 'good'): 65.5%

Threshold 0.5
  Accuracy : 71.0%
  Precision: 61.2%
  Recall   : 43.5%   (caught 30 of 69 bad customers)
  TP=30  FP=19  FN=39  TN=112

Training accuracy: 79.8%


**What I see:** on unseen customers the model is **71% accurate**, beating the 65.5% baseline of always saying "good". But it only **catches 43% of the bad customers** (30 of 69).
Training accuracy (~80%) is higher than test accuracy (71%), so the model is a bit **overfitting**: it learned some details of the training customers that don't generalize.

## 7. Choosing a better threshold
For a bank, **missing a bad customer (FN) is much worse** than rejecting a good one (FP). A lost loan costs far more than a lost customer. The dataset's authors suggest counting it as **5× worse**.
So we lower the threshold: we flag a customer as risky already at a 30% probability.

In [8]:
evaluate(0.3)

Threshold 0.3
  Accuracy : 66.5%
  Precision: 51.1%
  Recall   : 65.2%   (caught 45 of 69 bad customers)
  TP=45  FP=43  FN=24  TN=88


**What I see:** with threshold **0.3 the model catches 65% of the bad customers** (45 of 69) instead of 43%. Accuracy drops a little (66.5%), because we also reject more good customers. **This is a business decision, not just a math one:** the right threshold depends on what each mistake costs.

## 8. Looking inside the model: what drives risk?
Because the features were scaled, the weights can be compared directly. **Positive = more risk, negative = less risk.**

In [9]:
weights = pd.Series(w, index=feature_names).sort_values()

print("Strongest signals of LOW risk:")
print(weights.head(5).round(2))
print("\nStrongest signals of HIGH risk:")
print(weights.tail(5).round(2))

Strongest signals of LOW risk:
checking_status_no account        -0.52
credit_history_critical account   -0.33
purpose_used car                  -0.32
savings_>= 1000 DM                -0.32
employment_4-7 years              -0.28
dtype: float64

Strongest signals of HIGH risk:
credit_history_all paid at this bank    0.26
installment_rate                        0.30
purpose_new car                         0.31
checking_status_< 0 DM                  0.34
duration                                0.47
dtype: float64


**What I see:** the model learned the same patterns we found in notebook 03 by hand:
- **Higher risk:** long `duration`, a **negative bank account (< 0 DM)**, loans for **new cars**, and a high `installment_rate`
- **Lower risk:** **no checking account**, a "critical" credit history (the surprise from notebook 03), **used-car** loans, and **savings ≥ 1,000 DM**

## Summary of this notebook
1. Built **logistic regression from scratch in NumPy**: one-hot encoding, a train/test split, scaling without data leakage, sigmoid, cross-entropy loss and gradient descent.
2. **Test accuracy 71%** vs a 65.5% baseline. With **threshold 0.3 it catches 65% of the bad customers**.
3. The model's weights confirm the analysis: **loan duration and bank account balance** are the strongest risk signals.
4. **Next steps:** compare with scikit-learn's `LogisticRegression`, try regularization against the overfitting, and choose the threshold using the 5:1 cost ratio.